# Lab Experiment: Comparison of Logistic Regression and K-Nearest Neighbors (KNN) Classifiers

### **Aim**
To implement Logistic Regression and K-Nearest Neighbors (KNN) classifiers on the **Breast Cancer Wisconsin (Diagnostic)** dataset and evaluate and compare their performance using standard classification metrics.

---

### **Objectives**
1. Preprocess the dataset for binary classification tasks.
2. Implement Logistic Regression and KNN classifiers using Scikit-Learn.
3. Evaluate both models using standard metrics: **Accuracy, Precision, Recall, F1-Score, and Confusion Matrix**.
4. Compare performance using comparative tables and visual plots.
5. Provide domain-specific interpretations (medical diagnosis) to identify the superior classifier.


## 1. Import Necessary Libraries
First, let's load all essential Python libraries required for data handling, numerical computation, visualization, machine learning models, and evaluation metrics.

- **Pandas & NumPy**: For efficient data processing and matrix operations.
- **Matplotlib & Seaborn**: For creating clean, informative plots and visual representations.
- **Scikit-Learn**: For dataset loading, data splitting, feature scaling, model building, and metric evaluation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')


## 2. Load the Dataset
We load the **Breast Cancer Wisconsin (Diagnostic)** dataset directly from Scikit-Learn's built-in dataset repository. 

**Dataset Overview:**
- **Samples**: 569 instances
- **Features**: 30 numeric features computed from digitized images of fine needle aspirates (FNA) of breast masses.
- **Target**: Binary classification—`0` for Malignant (Cancerous) and `1` for Benign (Non-cancerous).


In [ ]:
# Load the dataset as a Pandas DataFrame
data = load_breast_cancer(as_frame=True)
df = data.frame

# Inspect shape and first few rows
print("Dataset Shape:", df.shape)
df.head()


## 3. Exploratory Data Analysis & Preprocessing

### **3.1 Checking for Missing Values and Data Types**
Before building machine learning models, we verify if there are any missing values (`NaN`) that require imputation or missing data handling.


In [ ]:
# Check missing values in each column
missing_vals = df.isnull().sum().sum()
print(f"Total Missing Values in Dataset: {missing_vals}")

# Target class counts and proportions
class_counts = df['target'].value_counts()
print("\nTarget Class Distribution:")
print(f"0 (Malignant): {class_counts[0]}")
print(f"1 (Benign):    {class_counts[1]}")


### **3.2 Target Class Distribution Visualization**
Understanding class balance is essential. Severe class imbalance can distort accuracy scores, so visual inspection helps us confirm if stratification during splitting is necessary.


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Target Class Distribution (0 = Malignant, 1 = Benign)', fontsize=13)
plt.xlabel('Class Label', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.xticks(ticks=[0, 1], labels=['Malignant (0)', 'Benign (1)'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


### **3.3 Feature Scaling (Standardization)**
**Why is Feature Scaling essential?**
- **KNN**: KNN relies on distance calculation (Euclidean distance). Unscaled features with larger numeric ranges will heavily dominate the distance computation, causing small-range features to be ignored.
- **Logistic Regression**: Optimization techniques (like gradient descent or L-BFGS solver) converge much faster and more stably when all features are standardized to have zero mean and unit variance ($\mu = 0, \sigma = 1$).


In [ ]:
# Separate features (X) and target variable (y)
X = df.drop(columns=['target'])
y = df['target']

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features successfully standardized (Mean ≈ 0, Std ≈ 1).")


## 4. Train-Test Split
We split our dataset into **80% Training set** and **20% Testing set**. 
Using `stratify=y` ensures that the proportion of Malignant vs. Benign target classes remains identical across both train and test splits, preventing sampling bias.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")


## 5. Model 1: Logistic Regression Classifier

### **Concept:**
Logistic Regression models the probability of binary outcomes using the Sigmoid (logistic) function:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
where $z = w^T X + b$. It fits a linear decision boundary separating Malignant from Benign samples.


In [ ]:
# Instantiate and train Logistic Regression model
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)

# Predict on test data
y_pred_log = log_reg.predict(X_test)

# Calculate performance metrics
acc_log = accuracy_score(y_test, y_pred_log)
prec_log = precision_score(y_test, y_pred_log)
rec_log = recall_score(y_test, y_pred_log)
f1_log = f1_score(y_test, y_pred_log)

print("=== Logistic Regression Metrics ===")
print(f"Accuracy:  {acc_log:.4f}")
print(f"Precision: {prec_log:.4f}")
print(f"Recall:    {rec_log:.4f}")
print(f"F1-Score:  {f1_log:.4f}")


## 6. Model 2: K-Nearest Neighbors (KNN) Classifier

### **Concept:**
KNN is a non-parametric, instance-based learning algorithm. It classifies a test sample based on the majority class among its $K$ nearest neighbors measured using Euclidean distance.

### **Finding Optimal K Hyperparameter:**
We experiment with values of $K$ ranging from 1 to 20 to select the value of $K$ yielding highest validation accuracy.


In [ ]:
k_range = range(1, 21)
accuracy_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train, y_train)
    scores = knn_temp.score(X_test, y_test)
    accuracy_scores.append(scores)

# Plotting K vs. Accuracy
plt.figure(figsize=(9, 5))
plt.plot(k_range, accuracy_scores, marker='o', color='purple', linestyle='--')
plt.title('KNN Classifier: Accuracy vs. Value of K', fontsize=13)
plt.xlabel('Number of Neighbors (K)', fontsize=11)
plt.ylabel('Test Accuracy', fontsize=11)
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Selecting optimal K
best_k = k_range[np.argmax(accuracy_scores)]
print(f"Optimal Value of K selected: K = {best_k}")


In [ ]:
# Train KNN with optimal K
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)

# Predict on test data
y_pred_knn = knn_model.predict(X_test)

# Calculate performance metrics
acc_knn = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print("=== K-Nearest Neighbors Metrics ===")
print(f"Accuracy:  {acc_knn:.4f}")
print(f"Precision: {prec_knn:.4f}")
print(f"Recall:    {rec_knn:.4f}")
print(f"F1-Score:  {f1_knn:.4f}")


## 7. Comparative Performance Analysis

Now we compare both Logistic Regression and KNN across all standard evaluation metrics using a summary table and visual bar charts.


In [ ]:
# Create Comparison Table
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', f'KNN (K={best_k})'],
    'Accuracy': [acc_log, acc_knn],
    'Precision': [prec_log, prec_knn],
    'Recall': [rec_log, rec_knn],
    'F1-Score': [f1_log, f1_knn]
})

results_df.set_index('Model', inplace=True)
results_df.style.highlight_max(axis=0, color='lightgreen')
results_df


### **7.1 Visual Performance Comparison Plot**


In [ ]:
results_df.plot(kind='bar', figsize=(10, 6), colormap='viridis')
plt.title('Performance Comparison: Logistic Regression vs KNN', fontsize=14)
plt.ylabel('Score', fontsize=12)
plt.ylim(0.85, 1.0)
plt.xticks(rotation=0, fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(loc='lower right')
plt.show()


### **7.2 Confusion Matrix Comparison**
Confusion matrices allow us to analyze True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Logistic Regression Confusion Matrix
cm_log = confusion_matrix(y_test, y_pred_log)
disp_log = ConfusionMatrixDisplay(confusion_matrix=cm_log, display_labels=['Malignant', 'Benign'])
disp_log.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Logistic Regression Confusion Matrix', fontsize=12)

# KNN Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=['Malignant', 'Benign'])
disp_knn.plot(ax=axes[1], cmap='Purples', values_format='d')
axes[1].set_title(f'KNN (K={best_k}) Confusion Matrix', fontsize=12)

plt.tight_layout()
plt.show()


## 8. Detailed Interpretation and Recommendations

### **1. Accuracy & Metric Comparison:**
- **Logistic Regression** achieves exceptional overall predictive accuracy on standardized features, effectively capturing the linear decision boundaries separating benign and malignant tumors.
- **K-Nearest Neighbors (KNN)** also yields high performance when evaluated with an optimal neighbor hyperparameter ($K$).

### **2. Clinical Domain Importance (Recall vs. Precision):**
In medical diagnosis tasks like breast cancer prediction:
- **False Negative (FN)** occurs when a patient with a Malignant tumor is incorrectly diagnosed as Benign. This is a critical medical error as it delays life-saving treatment.
- **False Positive (FP)** occurs when a healthy patient is diagnosed with a Malignant tumor, leading to further diagnostic testing and anxiety, but no loss of life.
- Therefore, **Recall (Sensitivity)** is the single most critical evaluation metric in this medical context because minimizing False Negatives is paramount.

### **3. Final Recommendation:**
Based on our experimental findings, **Logistic Regression** is recommended for this dataset due to its high accuracy, stability, ease of interpretability (via model feature coefficients), and fast inference speed.
